# pSMAD_2024-08-21 — 00_conversion

**Feeds:** ED Fig 2j, 2k

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Conversion

Convert the dataset-2 raw channel files to OME-TIFF outputs for the sibling workflow.

## Setup

In [ ]:
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
import tifffile
from IPython.display import display


In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent.resolve()

INPUT_DIR = ROOT / '2024-08-21_pSMAD'
OUT_DIR = ROOT / 'results/converted_tiff'
FORMAT = 'ome-tiff'

TOOLS_DIR = ROOT / 'tools/bftools'
PROJECT_BFCONVERT = TOOLS_DIR / 'bfconvert'
PROJECT_BIOFORMATS2RAW = TOOLS_DIR / 'bioformats2raw'

if FORMAT == 'ome-tiff' and PROJECT_BFCONVERT.exists():
    TOOL_PATH = PROJECT_BFCONVERT
elif FORMAT == 'ome-zarr' and PROJECT_BIOFORMATS2RAW.exists():
    TOOL_PATH = PROJECT_BIOFORMATS2RAW
else:
    TOOL_PATH = None

OVERWRITE = True
RUN_CONVERSION = True
DRY_RUN = False
RESOLUTIONS = 4

print('Project root:', ROOT)
print('Input dir:', INPUT_DIR)
print('Output dir:', OUT_DIR)
print('Tool path override:', TOOL_PATH)


## Conversion Tool Resolution

In [ ]:
def resolve_tool(format_name: str, explicit_tool: Path | None, dry_run: bool) -> str:
    if explicit_tool is not None:
        return str(explicit_tool)
    fallback = 'bfconvert' if format_name == 'ome-tiff' else 'bioformats2raw'
    found = shutil.which(fallback)
    if found:
        return found
    if dry_run:
        return fallback
    raise RuntimeError(f"Required tool '{fallback}' not found.")

def output_suffix(format_name: str) -> str:
    return '.ome.tif' if format_name == 'ome-tiff' else '.ome.zarr'

def build_command(tool: str, src: Path, dst: Path, format_name: str, overwrite: bool, resolutions: int) -> list[str]:
    if format_name == 'ome-tiff':
        cmd = [tool]
        if overwrite:
            cmd.append('-overwrite')
        cmd.extend([str(src), str(dst)])
        return cmd
    cmd = [tool, str(src), str(dst), '--resolutions', str(resolutions)]
    if overwrite:
        cmd.append('--overwrite')
    return cmd


## Save Stage Outputs

In [ ]:
if not INPUT_DIR.exists():
    raise RuntimeError(f'Missing input dir: {INPUT_DIR}')

OUT_DIR.mkdir(parents=True, exist_ok=True)
czi_files = sorted(INPUT_DIR.glob('*.czi'))
if not czi_files:
    raise RuntimeError(f'No .czi files found in {INPUT_DIR}')

suffix = output_suffix(FORMAT)
plan_rows = []
for src in czi_files:
    dst = OUT_DIR / f'{src.stem}{suffix}'
    plan_rows.append({
        'source_file': src.name,
        'source_size_mb': round(src.stat().st_size / (1024**2), 2),
        'dest_file': dst.name,
        'dest_exists': dst.exists(),
        'will_run': OVERWRITE or (not dst.exists()),
    })
plan_df = pd.DataFrame(plan_rows)
display(plan_df)


In [ ]:
if not RUN_CONVERSION:
    print('RUN_CONVERSION=False -> no conversion executed.')
else:
    tool = resolve_tool(FORMAT, TOOL_PATH, dry_run=DRY_RUN)
    print('Using tool:', tool)
    started = time.time()
    for src in czi_files:
        dst = OUT_DIR / f'{src.stem}{suffix}'
        if dst.exists() and not OVERWRITE:
            print(f'[SKIP] {dst} exists')
            continue
        cmd = build_command(tool, src, dst, FORMAT, OVERWRITE, RESOLUTIONS)
        print('[CMD]', ' '.join(cmd))
        if not DRY_RUN:
            subprocess.run(cmd, check=True)
            print(f'[OK] {src.name} -> {dst.name}')
    print(f'Done in {time.time() - started:.1f}s')


## Converted Output Review

In [ ]:
out_files = sorted(OUT_DIR.glob('*.ome.tif')) + sorted(OUT_DIR.glob('*.ome.zarr'))
rows = []
for p in out_files:
    rec = {'file': p.name, 'size_mb': round(p.stat().st_size / (1024**2), 2) if p.is_file() else None, 'series_count': None, 'series0_axes': None, 'series0_shape': None}
    if p.suffix.lower() in {'.tif', '.tiff'}:
        with tifffile.TiffFile(p) as tif:
            rec['series_count'] = len(tif.series)
            if tif.series:
                rec['series0_axes'] = tif.series[0].axes
                rec['series0_shape'] = tuple(int(x) for x in tif.series[0].shape)
    rows.append(rec)
display(pd.DataFrame(rows))
